## Baseline Model

In [1]:
# Load the cleaned Parquet file into a DataFrame

import pandas as pd

data = pd.read_parquet("../data/processed/train_transaction_cleaned.parquet", 
                       engine="fastparquet"
)


In [3]:
# Separate features and target variables

X = data.drop(columns=["isFraud", "TransactionID"])
y = data["isFraud"]

In [ ]:
# Identify numerical and categorical columns

numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Numerical columns:", len(numerical_columns))
print("Categorical columns:", len(categorical_columns))

Numerical columns: 323
Categorical columns: 14


In [8]:
# Use categorical columns with 50 or fewer unique values to avoid high cardinality issues in the baseline model.
usable_categorical_columns = [
    column
    for column in categorical_columns
    if X[column].nunique() <= 50
]

In [9]:
# Create the first feature set

selected_columns = numerical_columns + usable_categorical_columns
X = X[selected_columns]

In [10]:
# Create a chronological train-test split based on the TransactionDT column

split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_validation = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_validation = y.iloc[split_index:]

In [ ]:
# Build the preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Process numerical columns
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Process categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, usable_categorical_columns)
])

In [ ]:
# Train a baseline model using Logistic Regression

from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

In [13]:
# Train the baseline model

baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](335,)","['TransactionDT','TransactionAmt','card1',...,'M7','M8','M9']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,335
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrou

In [14]:
# Make predictions on the validation set

y_pred = baseline_model.predict(X_validation)
y_probability = baseline_model.predict_proba(X_validation)[:, 1]

In [ ]:
# Evaluate the baseline model

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, average_precision_score

print(classification_report(y_validation, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_pred))

print("ROC AUC Score:", roc_auc_score(y_validation, y_probability))

print("Average Precision:", average_precision_score(y_validation, y_probability))

              precision    recall  f1-score   support

           0       0.99      0.72      0.83    114044
           1       0.09      0.79      0.16      4064

    accuracy                           0.72    118108
   macro avg       0.54      0.75      0.50    118108
weighted avg       0.96      0.72      0.81    118108

Confusion Matrix:
[[82262 31782]
 [  863  3201]]
ROC AUC Score: 0.8294051288862261
Average Precision Score: 0.18898129349070125


## Baseline Model Summary

- Created a chronological 80/20 training and validation split.
- Used numerical and low-cardinality categorical features.
- Applied median imputation, scaling and one-hot encoding.
- Trained a class-weighted `Logistic Regression` baseline.
- Evaluated the model using precision, recall, F1, ROC-AUC and PR-AUC.
<br><br>
## Baseline Model Results

The Logistic Regression model was evaluated on 118,108 validation transactions.

### Fraud-Class Performance

- Precision: 0.09
- Recall: 0.79
- F1: 0.16
- ROC-AUC: 0.829
- PR-AUC / Average Precision: 0.189

### Confusion Matrix Results

- Correctly identified legitimate transactions: 82,262
- Legitimate transaction incorrectly flagged as fraudulent: 31,782
- Fraudulent transactions missed: 863
- Fraudulent transaction correctly flagged: 3,201

From this we can state that the model detected approximately 79% of fraudulent transactions. However, only about 9% iof the transactions flagged as fraudulent were actually fraud. This resulted in a high number of false-positives. 

This baseline demonstrates that the model can identify patterns to detect fraudulent transactions, but the precision and false-positive performance needs to be improved. This can be done with better feature engineering, tree-based models and threshold optimization.